# SHA3-256 PYNQ-Z2 reset 修正版驗證

先由管理者確認板子已完整恢復／重啟，並確認現行預約及你獨占的板號。將 bit、hwh、py、此 notebook 與 release_manifest.json 五個檔案放在新的同一資料夾。逐格執行並確認每格輸出。

## 1. 先測試 kernel

In [ ]:
print("kernel OK", flush=True)


## 2. 環境及驅動

In [ ]:
import hashlib
import platform
import pynq
from sha3_pynq import Sha3Session, self_test, validate_artifacts
print("PYNQ", pynq.__version__, "Python", platform.python_version(), flush=True)


## 3. 離線檔案檢查

這一格只讀取檔案，檢查 bit/hwh 配對雜湊、reset 極性、clock、DMA 位址與 AXI 接線。

In [ ]:
release = validate_artifacts("sha3_pynq.bit")
print(release, flush=True)


## 4. 下載 overlay 與檢查 DMA 啟動

本格會重新設定 FPGA。確認前一個已知可用範例已執行完畢，且這塊板正在由你使用。畫面依序顯示四個階段，便於定位停住的位置。

In [ ]:
session = Sha3Session("sha3_pynq.bit")
print("READY", session.overlay.ip_dict.keys(), flush=True)


## 5. abc 比對

In [ ]:
actual = session.hash(b"abc")
expected = hashlib.sha3_256(b"abc").digest()
print("FPGA  ", actual.hex())
print("Python", expected.hex())
assert actual == expected


## 6. 邊界與連續訊息

17 種訊息連跑兩輪，包含空訊息、135/136/137 bytes、272 bytes 與 4096 bytes。

In [ ]:
self_test(session, rounds=2)


## 7. 正常結束

In [ ]:
session.close()
print("buffers released")


## 錯誤恢復

若傳輸錯誤但 kernel 仍有回應，保存錯誤後可在新 cell 執行 session.reset()。新版會先要求兩個 DMA channel 停止，確認 halted 後才重新下載 overlay。若停止逾時，保留 buffer 並回報需要板端恢復。

若連 print 或網頁也無法回應，請管理者確認／重啟板子。Python timeout 無法中止已經卡住的底層硬體存取。記錄最後印出的階段及板號。

完成後下載測試結果，依實驗室規則清理自己的檔案。